# InfiniteScienceGym regeneration, part 1 of 2: prepare (and finish) on CPU

Runs on a **CPU** session (no GPU quota), with *Internet* on. Keep this notebook and its output
**private**: once repositories and questions are in it, it holds answers.

* `MODE = "prepare"`: pinned code (ISG and scidatasim), the model's weights at a recorded revision,
  a check that ISG's code, the tokenizer and the seed list work, and `plan.json` for the GPU
  notebook (`isg_generate.ipynb`). Save the version; add its output as an input of the GPU notebook.
* `MODE = "finish"`: add the GPU notebook's output(s) as inputs. Generates the questions (ISG's
  `2_generate_questions.py`, CPU) and runs the audit on the longest complete prefix of seeds
  (`audit_isg.py audit --complete-prefix`). Results in `results/`.

Steps and background: `docs/reproduce/isg_regeneration.md` in scidatasim.

In [ ]:
MODE = "prepare"          # "prepare" or "finish"
SDS_COMMIT = "main"       # scidatasim; pin the commit with patch 0064 for the paper
ISG_COMMIT = "2006f69872986e502074282ee4d23634e0cab689"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = ""       # empty: today's revision (recorded in plan.json)
VLLM_VERSION = ""         # empty: the newest (recorded in plan.json); the GPU notebook installs it
WEIGHTS_IN_OUTPUT = False # True: also keep the 8 GB of weights in this output (a fallback for the GPU notebook)
ISG_PACKAGES = ["numpy==2.4.0", "pandas==2.3.3", "scipy==1.17.0", "transformers==4.57.3", "XlsxWriter==3.2.9",
                "huggingface_hub<1.0"]
TORCH_CPU = "torch==2.9.1"
OUT = "/kaggle/working"

In [ ]:
import glob, json, os, shutil, subprocess, sys, time

def sh(cmd, **kw):
    """Run a shell command, show it, stop the notebook if it fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, **kw)

def isg_venv(packages, torch_cpu, path="/tmp/isg_venv"):
    """ISG's own versions in their own environment (ISG runs model-written code)."""
    py = f"{path}/bin/python"
    if not os.path.exists(py):
        sh(f"{sys.executable} -m pip install -q uv")
        sh(f"{sys.executable} -m uv venv -q --python 3.12 {path}")
        sh(f"{sys.executable} -m uv pip install -q --python {py} {torch_cpu} --index-url https://download.pytorch.org/whl/cpu")
        sh(f"{sys.executable} -m uv pip install -q --python {py} " + " ".join(f"'{p}'" for p in packages))
    return py

def entries(path):
    with open(path) as f:
        m = json.load(f)
    return len(m.get("seconds", {})) + len(m.get("failed", {}))

def gather(sources, fs, manifests=lambda name: True):
    """Finished repositories (<seed>.json) and manifests from attached outputs into fs;
    of two copies of a manifest, the one with more entries wins."""
    os.makedirs(fs, exist_ok=True)
    for path in sorted(sources):
        name, dest = os.path.basename(path), os.path.join(fs, os.path.basename(path))
        if name.startswith("regeneration_manifest") and manifests(name):
            if not os.path.exists(dest) or entries(path) > entries(dest):
                shutil.copy(path, dest)
        elif name.split(".")[0].isdigit() and name.endswith(".json") and not os.path.exists(dest):
            shutil.copy(path, dest)
    return len(glob.glob(os.path.join(fs, "[0-9]*.json")))

In [ ]:
code = f"{OUT}/code"
for name, url, commit in [("isg", "https://github.com/utahnlp/infinite-science-gym", ISG_COMMIT),
                          ("sds", "https://github.com/sandner-art/scientific-data-simulator", SDS_COMMIT)]:
    if not os.path.isdir(f"{code}/{name}"):
        sh(f"git clone -q {url} {code}/{name} && git -C {code}/{name} checkout -q {commit}")
commits = {n: subprocess.run(["git", "-C", f"{code}/{n}", "rev-parse", "HEAD"], capture_output=True,
                             text=True).stdout.strip() for n in ("isg", "sds")}
print(commits)
py = isg_venv(ISG_PACKAGES, TORCH_CPU)

In [ ]:
if MODE == "prepare":
    from huggingface_hub import HfApi, snapshot_download
    revision = HfApi().model_info(MODEL, revision=MODEL_REVISION or None).sha
    # the tokenizer and configs for the checks; the GPU notebook downloads the weights at this revision
    # itself (about a minute, against 15 minutes reading 8 GB from an input). WEIGHTS_IN_OUTPUT = True
    # also stores the weights here, as a fallback should the Hub be unreachable.
    snapshot_download(MODEL, revision=revision, local_dir=f"{OUT}/model",
                      allow_patterns=None if WEIGHTS_IN_OUTPUT else ["*.json", "*.txt"])
    if not VLLM_VERSION:  # the newest release, so every session installs the same one
        found = subprocess.run(f"{sys.executable} -m pip index versions vllm", shell=True, capture_output=True,
                               text=True).stdout
        VLLM_VERSION = found.split("(")[1].split(")")[0] if "(" in found else ""
    print("weights", MODEL, revision, "| vLLM", VLLM_VERSION or "(newest at install)")

In [ ]:
if MODE == "prepare":  # the checks that would otherwise cost GPU time
    tools = f"{code}/sds/tools/audit_isg.py"
    check = (f"import sys; sys.path[:0] = ['{code}/sds/tools', '{code}/isg']; import audit_isg as a; "
             f"import simulator; tok = a.load_tokenizer('{OUT}/model'); "
             "p = tok.apply_chat_template([{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'PRE:'}], "
             "tokenize=False, add_generation_prompt=False, continue_final_message=True); "
             "assert p.rstrip().endswith('PRE:'), p; print('ISG imports, tokenizer and template: fine')")
    sh(f"cd /tmp && {py} -c \"{check}\"")
    sh(f"{py} {tools} audit --isg {code}/isg --out {OUT}/results/seeds_only")   # seeds.txt check, no data
    keys = open(f"{code}/isg/seeds.txt").read().split()
    seeds = sorted({int(k.split(".")[0]) for k in keys})
    plan = {"model": MODEL, "revision": revision, "vllm": VLLM_VERSION, "commits": commits,
            "isg_packages": ISG_PACKAGES, "torch_cpu": TORCH_CPU, "repositories": len(seeds), "items": len(keys),
            "prepared": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())}
    json.dump(plan, open(f"{OUT}/plan.json", "w"), indent=1)
    print(json.dumps(plan, indent=1))

In [ ]:
if MODE == "finish":
    data = f"{OUT}/isgdata"
    n = gather(glob.glob("/kaggle/input/**/fs/*.json", recursive=True), f"{data}/fs")
    print(n, "repositories from the attached generation outputs")
    sh(f"cd {code}/isg && {py} scripts/2_generate_questions.py --cache-dir {data} --n-seeded-samples 5 "
       f"> {OUT}/questions.log 2>&1")
    sh(f"{py} {code}/sds/tools/audit_isg.py audit --isg {code}/isg --qa {data}/qa --complete-prefix "
       f"--out {OUT}/results")
    print(open(f"{OUT}/results/isg_audit.md").read())